# Загрузка данных и библиотек


## Импорт библиотек


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
from matplotlib import pyplot as plt

## Загрузка датасетов


In [ ]:
# Загружаем основной датасет заявок после EDA
app = pd.read_csv('../data/app_train_eda.csv')

# Загружаем историю предыдущих заявок клиентов
prev = pd.read_csv('../data/previous_application.csv')

# Загружаем данные о платежах по кредитам
pays = pd.read_csv('../data/installments_payments.csv')

balance = pd.read_csv('../data/POS_CASH_balance.csv')

# Feature Engineering из POS_CASH balance

In [ ]:
# Создаём копию для работы
balance_agg_prev = balance.copy()

# --- MONTHS_BALANCE ---
months_balance_agg = balance_agg_prev.groupby('SK_ID_PREV')['MONTHS_BALANCE'].agg([
    ('pos_months_balance_min', 'min'),
    ('pos_months_balance_max', 'max'),
    ('pos_months_balance_count', 'count'),
    ('pos_months_balance_last3', lambda x: (x >= -3).sum()),
    ('pos_months_balance_last6', lambda x: (x >= -6).sum()),
    ('pos_months_balance_last12', lambda x: (x >= -12).sum()),
]).reset_index()

# --- CNT_INSTALMENT ---
cnt_instalment_agg = balance_agg_prev.groupby('SK_ID_PREV')['CNT_INSTALMENT'].agg([
    ('pos_cnt_instalment_mean', 'mean'),
    ('pos_cnt_instalment_max', 'max'),
    ('pos_cnt_instalment_min', 'min'),
    ('pos_cnt_instalment_last', lambda x: x.iloc[-1] if len(x) > 0 else np.nan),
    ('pos_cnt_instalment_std', 'std'),
]).reset_index()

# --- CNT_INSTALMENT_FUTURE ---
cnt_instalment_future_agg = balance_agg_prev.groupby('SK_ID_PREV')['CNT_INSTALMENT_FUTURE'].agg([
    ('pos_cnt_instalment_future_mean', 'mean'),
    ('pos_cnt_instalment_future_max', 'max'),
    ('pos_cnt_instalment_future_min', 'min'),
    ('pos_cnt_instalment_future_last', lambda x: x.iloc[-1] if len(x) > 0 else np.nan),
    ('pos_cnt_instalment_future_zero_share', lambda x: (x == 0).sum() / len(x) if len(x) > 0 else 0),
    ('pos_cnt_instalment_future_zero_last', lambda x: 1 if (x.iloc[-1] == 0 if len(x) > 0 else False) else 0),
]).reset_index()

# --- NAME_CONTRACT_STATUS ---
status_last = balance_agg_prev.groupby('SK_ID_PREV')['NAME_CONTRACT_STATUS'].apply(
    lambda x: x.iloc[-1] if len(x) > 0 else 'Unknown'
).reset_index()
status_last.columns = ['SK_ID_PREV', 'pos_contract_status_last']

status_unique = balance_agg_prev.groupby('SK_ID_PREV')['NAME_CONTRACT_STATUS'].nunique().reset_index()
status_unique.columns = ['SK_ID_PREV', 'pos_contract_status_unique_count']

def get_status_stats(group):
    status_counts = group['NAME_CONTRACT_STATUS'].value_counts()
    status_share = group['NAME_CONTRACT_STATUS'].value_counts(normalize=True)
    result = {}
    for status in status_counts.index:
        result[f'pos_contract_status_months_{status}'] = status_counts[status]
        result[f'pos_contract_status_share_{status}'] = status_share[status]
    return pd.Series(result)

status_details = balance_agg_prev.groupby('SK_ID_PREV').apply(get_status_stats).reset_index()

# One-hot агрегация (sum и mean по статусам)
for status in balance_agg_prev['NAME_CONTRACT_STATUS'].unique():
    mask = balance_agg_prev['NAME_CONTRACT_STATUS'] == status
    status_balance_agg = balance_agg_prev[mask].groupby('SK_ID_PREV')['MONTHS_BALANCE'].agg(['sum', 'mean']).reset_index()
    status_balance_agg.columns = [
        'SK_ID_PREV',
        f'pos_contract_status_{status}_months_sum',
        f'pos_contract_status_{status}_months_mean'
    ]
    months_balance_agg = months_balance_agg.merge(status_balance_agg, on='SK_ID_PREV', how='left')

# --- SK_DPD ---
dpd_agg = balance_agg_prev.groupby('SK_ID_PREV')['SK_DPD'].agg([
    ('pos_sk_dpd_mean', 'mean'),
    ('pos_sk_dpd_max', 'max'),
    ('pos_sk_dpd_sum', 'sum'),
    ('pos_sk_dpd_std', 'std'),
    ('pos_sk_dpd_last', lambda x: x.iloc[-1] if len(x) > 0 else np.nan),
    ('pos_sk_dpd_count_gt0', lambda x: (x > 0).sum()),
    ('pos_sk_dpd_count_gt30', lambda x: (x > 30).sum()),
    ('pos_sk_dpd_count_gt90', lambda x: (x > 90).sum()),
    ('pos_sk_dpd_share_gt0', lambda x: (x > 0).sum() / len(x) if len(x) > 0 else 0),
    ('pos_sk_dpd_share_gt30', lambda x: (x > 30).sum() / len(x) if len(x) > 0 else 0),
    ('pos_sk_dpd_share_gt90', lambda x: (x > 90).sum() / len(x) if len(x) > 0 else 0),
]).reset_index()

# --- SK_DPD_DEF ---
dpd_def_agg = balance_agg_prev.groupby('SK_ID_PREV')['SK_DPD_DEF'].agg([
    ('pos_sk_dpd_def_mean', 'mean'),
    ('pos_sk_dpd_def_max', 'max'),
    ('pos_sk_dpd_def_sum', 'sum'),
    ('pos_sk_dpd_def_std', 'std'),
    ('pos_sk_dpd_def_last', lambda x: x.iloc[-1] if len(x) > 0 else np.nan),
    ('pos_sk_dpd_def_count_gt0', lambda x: (x > 0).sum()),
    ('pos_sk_dpd_def_count_gt30', lambda x: (x > 30).sum()),
    ('pos_sk_dpd_def_count_gt90', lambda x: (x > 90).sum()),
    ('pos_sk_dpd_def_share_gt0', lambda x: (x > 0).sum() / len(x) if len(x) > 0 else 0),
    ('pos_sk_dpd_def_share_gt30', lambda x: (x > 30).sum() / len(x) if len(x) > 0 else 0),
    ('pos_sk_dpd_def_share_gt90', lambda x: (x > 90).sum() / len(x) if len(x) > 0 else 0),
]).reset_index()

# Объединяем все агрегации по SK_ID_PREV
prev_agg_first = months_balance_agg.copy()
prev_agg_first = prev_agg_first.merge(cnt_instalment_agg, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(cnt_instalment_future_agg, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(status_last, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(status_unique, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(status_details, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(dpd_agg, on='SK_ID_PREV', how='left')
prev_agg_first = prev_agg_first.merge(dpd_def_agg, on='SK_ID_PREV', how='left')

print(f"Aggregated by SK_ID_PREV: {prev_agg_first.shape}")

## Агрегация POS_CASH по клиентам

После агрегации по кредитам (SK_ID_PREV) объединяем признаки с таблицей предыдущих заявок и агрегируем по клиентам (SK_ID_CURR).


In [ ]:

# Добавляем признаки уровня кредита в таблицу заявок для дальнейшей агрегации по клиенту
prev = prev.merge(prev_agg_first, on='SK_ID_PREV', how='left')

print(f"Prev after POS_CASH merge: {prev.shape}")

### Агрегация POS_CASH признаков по клиенту

Усредняем, суммируем и находим экстремумы POS_CASH признаков по каждому клиенту.


In [ ]:
# Список колонок POS_CASH (все, кроме SK_ID_PREV)
pos_prev_cols = [c for c in prev_agg_first.columns if c != 'SK_ID_PREV']

# Агрегируем каждый признак по клиенту (mean, max, min, sum, std)
pos_cash_client = (
    prev
    .groupby('SK_ID_CURR')[pos_prev_cols]
    .agg(['mean', 'max', 'min', 'sum', 'std'])
)

pos_cash_client.columns = ['POS_' + '_'.join(col) for col in pos_cash_client.columns]

# Количество POS-кредитов клиента
pos_cash_client['POS_CREDIT_COUNT'] = prev.groupby('SK_ID_CURR')['SK_ID_PREV'].nunique()

# Объединяем с основным датасетом
app = app.merge(pos_cash_client, how='left', left_on='SK_ID_CURR', right_index=True)

print(f"App after POS_CASH client aggregation: {app.shape}")

### Дополнительные признаки на уровне клиента

Рассчитываем мета-признаки: активные/завершённые кредиты, просрочки, сроки кредитов.


In [ ]:
# ============================================================================
# ШАГ 4: Дополнительные клиентские признаки на основе POS_CASH
# ============================================================================

# --- Активные и завершённые кредиты ---
# Определяем последний статус каждого кредита
last_status = (
    balance
    .groupby('SK_ID_PREV')['NAME_CONTRACT_STATUS']
    .last()
    .reset_index()
)
last_status = last_status.merge(
    prev[['SK_ID_PREV', 'SK_ID_CURR']], on='SK_ID_PREV', how='left'
)

app['POS_ACTIVE_CREDITS'] = (
    last_status[last_status['NAME_CONTRACT_STATUS'] != 'Completed']
    .groupby('SK_ID_CURR')
    .size()
)
app['POS_COMPLETED_CREDITS'] = (
    last_status[last_status['NAME_CONTRACT_STATUS'] == 'Completed']
    .groupby('SK_ID_CURR')
    .size()
)
app[['POS_ACTIVE_CREDITS', 'POS_COMPLETED_CREDITS']] = (
    app[['POS_ACTIVE_CREDITS', 'POS_COMPLETED_CREDITS']].fillna(0).astype(int)
)

# --- Средний срок кредита (размах MONTHS_BALANCE) ---
credit_term = (
    balance
    .groupby('SK_ID_PREV')['MONTHS_BALANCE']
    .agg(lambda x: x.max() - x.min())
    .reset_index(name='pos_credit_term')
)
credit_term = credit_term.merge(
    prev[['SK_ID_PREV', 'SK_ID_CURR']], on='SK_ID_PREV', how='left'
)
app['POS_AVG_CREDIT_TERM'] = credit_term.groupby('SK_ID_CURR')['pos_credit_term'].mean()

# --- Просрочки на уровне клиента ---
# Максимальная просрочка среди всех кредитов
max_dpd_per_credit = (
    balance
    .groupby('SK_ID_PREV')['SK_DPD']
    .max()
    .reset_index()
)
max_dpd_per_credit = max_dpd_per_credit.merge(
    prev[['SK_ID_PREV', 'SK_ID_CURR']], on='SK_ID_PREV', how='left'
)
app['POS_MAX_DPD_ALL'] = max_dpd_per_credit.groupby('SK_ID_CURR')['SK_DPD'].max()

# Средняя просрочка среди всех кредитов
app['POS_AVG_DPD_ALL'] = max_dpd_per_credit.groupby('SK_ID_CURR')['SK_DPD'].mean()

# --- Количество и доля кредитов с просрочкой ---
max_dpd_per_credit['POS_HAS_DPD'] = (max_dpd_per_credit['SK_DPD'] > 0).astype(int)
app['POS_DPD_CREDIT_COUNT'] = (
    max_dpd_per_credit.groupby('SK_ID_CURR')['POS_HAS_DPD'].sum()
)
app['POS_DPD_CREDIT_RATIO'] = (
    app['POS_DPD_CREDIT_COUNT'] / app['POS_CREDIT_COUNT'].replace(0, np.nan)
)

# --- Среднее количество оставшихся платежей ---
last_remaining = (
    balance
    .groupby('SK_ID_PREV')['CNT_INSTALMENT_FUTURE']
    .last()
    .reset_index()
)
last_remaining = last_remaining.merge(
    prev[['SK_ID_PREV', 'SK_ID_CURR']], on='SK_ID_PREV', how='left'
)
app['POS_AVG_REMAINING_INSTALMENTS'] = (
    last_remaining.groupby('SK_ID_CURR')['CNT_INSTALMENT_FUTURE'].mean()
)

# --- Количество полностью погашенных кредитов ---
fully_paid = last_remaining[last_remaining['CNT_INSTALMENT_FUTURE'] == 0]
app['POS_FULLY_PAID_COUNT'] = fully_paid.groupby('SK_ID_CURR').size()
app['POS_FULLY_PAID_COUNT'] = app['POS_FULLY_PAID_COUNT'].fillna(0).astype(int)

print("Дополнительные клиентские признаки POS_CASH добавлены")

# Feature Engineering из Installment Payments

Installment payments.csv содержит информацию о платежах по каждому кредиту. На основе этих данных рассчитаем агрегированные признаки, характеризующие платёжную дисциплину клиента, и объединим их с таблицей previous_application.


## Версии договора

Рассчитаем количество версий договора по каждому кредиту.


In [ ]:
# Максимальная версия договора по каждому кредиту
prev = prev.merge(
    pays.groupby('SK_ID_PREV')['NUM_INSTALMENT_VERSION'].max().reset_index().rename(
        columns={'NUM_INSTALMENT_VERSION': 'max_instalment_version'}
    ),
    on='SK_ID_PREV',
    how='left'
)

# Количество уникальных версий договора по каждому кредиту
prev = prev.merge(
    pays.groupby('SK_ID_PREV')['NUM_INSTALMENT_VERSION'].nunique().reset_index().rename(
        columns={'NUM_INSTALMENT_VERSION': 'nunique_instalment_version'}
    ),
    on='SK_ID_PREV',
    how='left'
)

In [ ]:
# Максимальный номер платежа по каждому кредиту
prev = prev.merge(
    pays.groupby('SK_ID_PREV')['NUM_INSTALMENT_NUMBER'].max().reset_index().rename(
        columns={'NUM_INSTALMENT_NUMBER': 'max_instalment_number'}
    ),
    on='SK_ID_PREV',
    how='left'
)

## Анализ платёжного поведения

Создадим признаки, характеризующие своевременность и полноту платежей.


### Своевременность платежей


In [ ]:
# Флаг: платёж совершён вовремя (день оплаты <= дня, в который нужно было оплатить)
pays['is_paid_on_time'] = (pays['DAYS_INSTALMENT'] > pays['DAYS_ENTRY_PAYMENT']).astype(int)

In [ ]:
# Доля платежей, оплаченных вовремя, по каждому кредиту
prev = prev.merge(
    pays.groupby('SK_ID_PREV')['is_paid_on_time'].mean().reset_index().rename(
        columns={'is_paid_on_time': 'mean_is_paid_on_time'}
    ),
    on='SK_ID_PREV',
    how='left'
)

### Просроченные платежи


In [ ]:
# Количество дней просрочки (если платёж был позже положенного срока)
pays['days_late'] = pays['DAYS_ENTRY_PAYMENT'] - pays['DAYS_INSTALMENT']
pays['days_late'] = pays['days_late'].clip(lower=0)

In [ ]:
# Среднее и максимальное количество дней просрочки по каждому кредиту
days_late_stats = (
    pays
    .groupby('SK_ID_PREV')['days_late']
    .agg(['mean', 'max'])
    .rename(columns={'mean': 'days_late_mean', 'max': 'days_late_max'})
    .reset_index()
)

prev = prev.merge(days_late_stats, on='SK_ID_PREV', how='left')

### Досрочные платежи


In [ ]:
# Количество дней досрочной оплаты (если платёж был раньше срока)
pays['days_early'] = pays['DAYS_INSTALMENT'] - pays['DAYS_ENTRY_PAYMENT']
pays['days_early'] = pays['days_early'].clip(lower=0)

In [ ]:
# Среднее и максимальное количество дней досрочной оплаты по каждому кредиту
days_early_stats = (
    pays
    .groupby('SK_ID_PREV')['days_early']
    .agg(['mean', 'max'])
    .rename(columns={'mean': 'days_early_mean', 'max': 'days_early_max'})
    .reset_index()
)

prev = prev.merge(days_early_stats, on='SK_ID_PREV', how='left')

### Размер платежа относительно обязательного


In [ ]:
# Отношение фактического платежа к обязательному
pays['payment_ratio'] = pays['AMT_PAYMENT'] / pays['AMT_INSTALMENT'].replace(0, np.nan)

# Статистики отношения платежа к обязательному по каждому кредиту
payment_ratio_stats = (
    pays
    .groupby('SK_ID_PREV')['payment_ratio']
    .agg(['mean', 'max', 'std', 'min'])
    .rename(columns={
        'mean': 'payment_ratio_mean',
        'max': 'payment_ratio_max',
        'std': 'payment_ratio_std',
        'min': 'payment_ratio_min'
    })
    .reset_index()
)

prev = prev.merge(payment_ratio_stats, on='SK_ID_PREV', how='left')

## Недоплаты

Рассчитаем признаки, связанные с неполной оплатой платежей.


In [ ]:
# Сумма недоплаты (если платеж меньше обязательного)
pays['unpaid_amount'] = (pays['AMT_INSTALMENT'] - pays['AMT_PAYMENT']).clip(lower=0)

# Флаг наличия недоплаты
pays['is_unpaid'] = (pays['unpaid_amount'] > 0).astype(int)

# Доля платежей с недоплатой по каждому кредиту
prev = prev.merge(
    pays.groupby('SK_ID_PREV')['is_unpaid'].mean().reset_index().rename(
        columns={'is_unpaid': 'unpaid_ratio'}
    ),
    on='SK_ID_PREV',
    how='left'
)

# Статистики суммы недоплаты по каждому кредиту
unpaid_amount_stats = (
    pays
    .groupby('SK_ID_PREV')['unpaid_amount']
    .agg(['mean', 'max', 'min', 'sum', 'std'])
    .rename(columns={
        'mean': 'unpaid_amount_mean',
        'min': 'unpaid_amount_min',
        'max': 'unpaid_amount_max',
        'sum': 'unpaid_amount_sum',
        'std': 'unpaid_amount_std'
    })
    .reset_index()
)

prev = prev.merge(unpaid_amount_stats, on='SK_ID_PREV', how='left')

## Переплаты

Рассчитаем признаки, связанные с переплатой по кредиту.


In [ ]:
# Сумма переплаты (если платеж больше обязательного)
pays['overpaid_amount'] = (pays['AMT_PAYMENT'] - pays['AMT_INSTALMENT']).clip(lower=0)

# Статистики суммы переплаты по каждому кредиту
overpaid_amount_stats = (
    pays
    .groupby('SK_ID_PREV')['overpaid_amount']
    .agg(['mean', 'max', 'sum'])
    .rename(columns={
        'mean': 'overpaid_amount_mean',
        'max': 'overpaid_amount_max',
        'sum': 'overpaid_amount_sum'
    })
    .reset_index()
)

prev = prev.merge(overpaid_amount_stats, on='SK_ID_PREV', how='left')

### Агрегация признаков из installment payments по клиентам

Агрегируем созданные на основе installment payments признаки на уровне клиента (SK_ID_CURR) и объединяем с основным датасетом.


In [ ]:
# Список признаков из installment payments, добавленных в prev
installment_cols = [
    'max_instalment_version',
    'nunique_instalment_version',
    'max_instalment_number',
    'mean_is_paid_on_time',
    'days_late_mean', 'days_late_max',
    'days_early_mean', 'days_early_max',
    'payment_ratio_mean', 'payment_ratio_max', 'payment_ratio_std', 'payment_ratio_min',
    'unpaid_ratio',
    'unpaid_amount_mean', 'unpaid_amount_min', 'unpaid_amount_max', 'unpaid_amount_sum', 'unpaid_amount_std',
    'overpaid_amount_mean', 'overpaid_amount_max', 'overpaid_amount_sum',
]

# Агрегируем по клиенту: среднее, максимум, минимум, сумма
installment_agg = (
    prev
    .groupby('SK_ID_CURR')[installment_cols]
    .agg(['mean', 'max', 'min', 'sum'])
)

installment_agg.columns = ['INST_' + '_'.join(col) for col in installment_agg.columns]

app = app.merge(installment_agg, how='left', left_on='SK_ID_CURR', right_index=True)

# Feature Engineering из Previous Application


## Исследование предыдущих заявок

Рассмотрим, сколько заявок у клиентов было до подачи последней.


In [ ]:
# Строим гистограмму распределения количества предыдущих заявок на клиента
prev.groupby('SK_ID_CURR')['SK_ID_PREV'].count().hist(bins=100)

Видно, что как минимум 60 тысяч клиентов ранее подавали заявку на кредит. Из их кредитной истории можно извлечь множество полезных признаков.


## Базовые признаки на основе предыдущих заявок


In [ ]:
# Если нет сведений о первоначальном взносе, считаем его равным 0
prev['AMT_DOWN_PAYMENT'] = prev['AMT_DOWN_PAYMENT'].fillna(0)

# Наиболее частый день недели подачи заявки для каждого клиента
app['WEEKDAY_APPR_PROCESS_START_most_freq'] = (
    prev
    .groupby('SK_ID_CURR')['WEEKDAY_APPR_PROCESS_START']
    .agg(lambda x: x.value_counts().idxmax())
)

### Количество предыдущих заявок


In [ ]:
# Общее количество предыдущих заявок для каждого клиента
count_loan = Counter(prev['SK_ID_CURR'])
app['prev_loan_count'] = app['SK_ID_CURR'].map(count_loan).fillna(0).astype(int)

### Соотношения признаков внутри заявки

Создадим относительные признаки на основе полей заявки.


In [ ]:
# Отношение суммы кредита к запрашиваемой сумме
prev['credit_to_app_ratio'] = prev['AMT_CREDIT'] / prev['AMT_APPLICATION']
# Доля первоначального взноса относительно суммы кредита
prev['downpayment_ratio'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
# Отношение стоимости товара к сумме кредита
prev['goods_credit_ratio'] = prev['AMT_GOODS_PRICE'] / prev['AMT_CREDIT']
# Отношение аннуитетного платежа к сумме кредита
prev['annuity_credit_ratio'] = prev['AMT_ANNUITY'] / prev['AMT_CREDIT']

## Агрегация числовых признаков

Агрегируем числовые характеристики предыдущих заявок по каждому клиенту.


In [ ]:
# Список числовых признаков для агрегации
num_cols = [
    'AMT_CREDIT', 'AMT_APPLICATION', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CNT_PAYMENT', 'DAYS_DECISION', 'SELLERPLACE_AREA',
    'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE', 'DAYS_TERMINATION',
    'credit_to_app_ratio', 'downpayment_ratio',
    'goods_credit_ratio', 'annuity_credit_ratio',
]

# Группируем по клиенту и вычисляем статистики по каждому числовому признаку
prev_agg = (
    prev
    .groupby('SK_ID_CURR')[num_cols]
    .agg(['mean', 'max', 'min', 'std', 'sum'])
)

# Формируем названия колонок: признак_статистика
prev_agg.columns = ['_'.join(col) for col in prev_agg.columns]

# Добавляем общее количество предыдущих заявок
prev_agg['TOTAL_PREV_COUNT'] = prev.groupby('SK_ID_CURR').size()

# Объединяем агрегированные признаки с основным датасетом
app = app.merge(prev_agg, how='left', left_on='SK_ID_CURR', right_index=True)

### Статусы заявок

Рассчитываем долю заявок каждого статуса (одобрена, отказано, отменена и т.д.) для каждого клиента.


In [ ]:
# Смотрим распределение статусов заявок
prev['NAME_CONTRACT_STATUS'].value_counts()

# Строим кросс-таблицу: клиент x статус заявки
status = pd.crosstab(prev['SK_ID_CURR'], prev['NAME_CONTRACT_STATUS'])
status['TOTAL_PREV_COUNT'] = status.sum(axis=1)

# Вычисляем долю каждого статуса по каждому клиенту
for col_name in ['Approved', 'Refused', 'Canceled', 'Unused offer']:
    status[f'{col_name.upper()}_RATIO'] = (
        status[col_name] / status['TOTAL_PREV_COUNT']
    )

status = status.drop(columns=['TOTAL_PREV_COUNT'])

app = app.merge(status, how='left', left_on='SK_ID_CURR', right_index=True)

### Агрегация категориальных признаков

Для каждого клиента рассчитываем долю заявок по каждой категории.


In [ ]:
# Список категориальных признаков для агрегации
cat_cols = [
    'CODE_REJECT_REASON',      # Причина отказа
    'NAME_CONTRACT_TYPE',      # Тип контракта
    'NAME_PORTFOLIO',          # Портфель
    'NAME_CLIENT_TYPE',        # Тип клиента
    'NAME_YIELD_GROUP',        # Группа доходности
    'CHANNEL_TYPE',            # Канал продаж
    'NAME_PAYMENT_TYPE',       # Тип платежа
]

for col in cat_cols:
    # Строим кросс-таблицу: клиент x категория
    crosstab = pd.crosstab(prev['SK_ID_CURR'], prev[col])
    total = crosstab.sum(axis=1).replace(0, np.nan)
    # Вычисляем долю каждой категории для каждого клиента
    ratios = crosstab.div(total, axis=0)
    ratios = ratios.add_prefix(f'{col}_').add_suffix('_RATIO')
    app = app.merge(ratios, how='left', left_on='SK_ID_CURR', right_index=True)

### Признаки последней заявки

Извлекаем информацию о последней по времени заявке для каждого клиента.


In [ ]:
# Выбираем последнюю заявку для каждого клиента (по DAYS_DECISION)
last_app = prev.loc[prev.groupby('SK_ID_CURR')['DAYS_DECISION'].idxmax()]

# Отбираем ключевые поля последней заявки
last_features = last_app[[
    'SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON',
    'NAME_CLIENT_TYPE', 'AMT_CREDIT', 'AMT_APPLICATION', 'DAYS_DECISION'
]].copy()

last_features = last_features.rename(columns={
    'NAME_CONTRACT_STATUS': 'LAST_STATUS',
    'CODE_REJECT_REASON': 'LAST_REJECT_REASON',
    'NAME_CLIENT_TYPE': 'LAST_CLIENT_TYPE',
    'AMT_CREDIT': 'LAST_AMT_CREDIT',
    'AMT_APPLICATION': 'LAST_AMT_APPLICATION',
    'DAYS_DECISION': 'LAST_DAYS_DECISION',
})

app = app.merge(last_features, how='left', on='SK_ID_CURR')

### Временные признаки

Создаём признаки, связанные со временем подачи заявок и интервалами между ними.


In [ ]:
# Размах дат принятия решений по заявкам клиента
app['DAYS_DECISION_range'] = app['DAYS_DECISION_max'] - app['DAYS_DECISION_min']

# Средний интервал между решениями
app['DAYS_DECISION_mean_gap'] = (
    app['DAYS_DECISION_range'] / app['TOTAL_PREV_COUNT'].replace(0, np.nan)
)

# Количество заявок за последние 6, 12 и 24 месяца
now = 0
for months, label in [(6, '6M'), (12, '12M'), (24, '24M')]:
    days = -months * 30
    count = (
        prev[prev['DAYS_DECISION'] > days]
        .groupby('SK_ID_CURR').size()
    )
    app[f'prev_count_last_{label}'] = count

### Флаги и пропуски как сигнал

Извлекаем информацию из флагов и пропущенных значений, которые могут быть полезным сигналом.


In [ ]:
# FLAG_LAST_APPL_PER_CONTRACT — доля заявок, являющихся последними по контракту
flag_crosstab = pd.crosstab(
    prev['SK_ID_CURR'], prev['FLAG_LAST_APPL_PER_CONTRACT']
)
total_flag = flag_crosstab.sum(axis=1).replace(0, np.nan)
flag_ratios = flag_crosstab.div(total_flag, axis=0)
flag_ratios = flag_ratios.add_prefix('FLAG_LAST_APPL_PER_CONTRACT_')
app = app.merge(flag_ratios, how='left', left_on='SK_ID_CURR', right_index=True)

# NFLAG_INSURED_ON_APPROVAL — доля застрахованных заявок среди всех заявок клиента
insured = prev[['SK_ID_CURR', 'NFLAG_INSURED_ON_APPROVAL']].copy()
insured['NFLAG_INSURED_ON_APPROVAL'] = insured['NFLAG_INSURED_ON_APPROVAL'].fillna(0)
app['NFLAG_INSURED_ON_APPROVAL_RATIO'] = (
    insured.groupby('SK_ID_CURR')['NFLAG_INSURED_ON_APPROVAL'].mean()
)

# NFLAG_LAST_APPL_IN_DAY — доля заявок, бывших последними в день подачи
app['NFLAG_LAST_APPL_IN_DAY_RATIO'] = (
    prev.groupby('SK_ID_CURR')['NFLAG_LAST_APPL_IN_DAY'].mean()
)

# HAS_DOWN_PAYMENT — был ли хотя бы один первоначальный взнос у клиента
app['HAS_DOWN_PAYMENT'] = (
    prev.groupby('SK_ID_CURR')['AMT_DOWN_PAYMENT']
    .apply(lambda x: x.notna().any())
    .astype(int)
)

# HAS_DAYS_* — наличие полей (отсутствуют для неодобренных заявок, поэтому их наличие — сигнал)
drawing_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
for col in drawing_cols:
    app[f'HAS_{col}'] = (
        prev.groupby('SK_ID_CURR')[col]
        .apply(lambda x: x.notna().any())
        .astype(int)
    )

## Обработка выбросов


In [ ]:
# Ограничиваем доход на уровне 99-го перцентиля
upper = app['AMT_INCOME_TOTAL'].quantile(0.99)
app['AMT_INCOME_TOTAL'] = app['AMT_INCOME_TOTAL'].clip(upper=upper)

# Удаляем аномальные записи с максимальным DAYS_EMPLOYED
app = app[app['DAYS_EMPLOYED'] != app['DAYS_EMPLOYED'].max()]

## Признаки по одобренным заявкам

Выделяем отдельную агрегацию только по одобренным заявкам — это даёт более чистую картину кредитной истории клиента.


In [ ]:
# Фильтруем только одобренные заявки
approved = prev[prev['NAME_CONTRACT_STATUS'] == 'Approved'].copy()

# Соотношения признаков для одобренных заявок
approved['credit_to_app_ratio'] = approved['AMT_CREDIT'] / approved['AMT_APPLICATION']
approved['downpayment_ratio'] = approved['AMT_DOWN_PAYMENT'] / approved['AMT_CREDIT']
approved['goods_credit_ratio'] = approved['AMT_GOODS_PRICE'] / approved['AMT_CREDIT']
approved['annuity_credit_ratio'] = approved['AMT_ANNUITY'] / approved['AMT_CREDIT']

# Агрегация числовых признаков по одобренным заявкам
approved_agg = (
    approved
    .groupby('SK_ID_CURR')[num_cols]
    .agg(['mean', 'max', 'min', 'std', 'sum'])
)

approved_agg.columns = ['APPROVED_' + '_'.join(col) for col in approved_agg.columns]

app = app.merge(approved_agg, how='left', left_on='SK_ID_CURR', right_index=True)

## Обработка пропусков


In [ ]:
# Заполняем возраст автомобиля нулём (нет машины — нет возраста)
app['OWN_CAR_AGE'] = app['OWN_CAR_AGE'].fillna(0)

## Сохранение результата


In [ ]:
# Сохраняем датасет с новыми признаками
app.to_csv('../data/app_feature_engineered.csv', index=False)

# Заключение

В этом ноутбуке:
- Выполнен фиче-инжиниринг на основе таблиц installments_payments и previous_application.
- Созданы признаки платёжной дисциплины, агрегации по клиентам, временные и категориальные признаки.
- Обработаны выбросы и пропуски при помощи эвристик.

Оставшиеся пропуски будут обработаны в препроцессоре (заполнение NaN и кодировка категориальных признаков) во избежание утечки данных.
